In [1]:
import operator
import random
from typing import Annotated, Sequence, TypedDict

from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, ToolMessage

from langgraph.managed.is_last_step import RemainingSteps
from langgraph.prebuilt import create_react_agent
from langgraph.errors import GraphRecursionError

/Users/aravindnatarajan/agents/lib/python3.14/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
EMBEDDING_MODEL = 'nomic-embed-text:latest'
LOCAL_LLM = 'gemma4:12b-mlx'
TEMPERATURE = 0.7

# Travel info store.
SAVE_DIR = './chroma_travel_db'

WEATHER: Sequence[str] = ['sunny', 'foggy', 'rainy', 'windy']

In [3]:
llm_model = ChatOllama(
    model=LOCAL_LLM,
    temperature=TEMPERATURE,
    use_responses_api=True
)

In [4]:
vectorstore_client = Chroma(
    persist_directory=SAVE_DIR,
    embedding_function=OllamaEmbeddings(model=EMBEDDING_MODEL)
)
retriever = vectorstore_client.as_retriever()

In [5]:
class WeatherForecast(TypedDict):
    town: str
    weather: Literal = WEATHER
    temperature: int
    
@tool(description='Get the weather forecast given the town name.')
def weather_forecast(town: str) -> dict:
    '''Get a weather forecast for a given town.
    Returns a WeatherForecast object with weather and temperature.
    '''
    _weather_options = ['sunny', 'foggy', 'rainy', 'windy']
    _temp_min = 18
    _temp_max = 31
    
    weather = random.choice(_weather_options)
    temperature = random.randint(_temp_min, _temp_max)
    return WeatherForecast(town=town, weather=weather, temperature=temperature)
    
@tool(description='Search travel information about destinations in England.')
def search_travel_info(query: str) -> str:
    """Search embedded WikiVoyage content for 
    information about destinations in England."""    
    docs = retriever.invoke(query)
    top = docs[:4] if isinstance(docs, list) else docs
    return "\n---\n".join(d.page_content for d in top)

tools = [weather_forecast, search_travel_info]    

In [6]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps

In [7]:
system_prompt = '''
You are a helpful travel assistant that searches information and retrieves weather forecasts.    

CRITICAL RULES:
    1. Only suggest destinations that you have found inside the 'search_travel_info' tool.
    2. Identify candidate towns from your travel info search and check the weather for MULTIPLE candidate towns in parallel (simultaneously) to find the ones with the best weather.
    3. If your initial batch of towns has bad weather, query the weather for any backup towns in a single batch before formulating your final answer.
'''
travel_info_agent = create_react_agent(
    model=llm_model,
    tools=tools,
    state_schema=AgentState,
    prompt=system_prompt,
)

/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_1863/3786618434.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  travel_info_agent = create_react_agent(


In [8]:
def chat_loop():
    print("UK Travel Assistant (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        state = {"messages": [HumanMessage(content=user_input)]}
        
        # 10 steps is more than enough for a search + weather fallback batch
        config = {"recursion_limit": 10} 
        
        try:
            result = travel_info_agent.invoke(state, config=config)
            print("\n\n\n")
            print(result)
            print("\n\n\n")
            response_msg = result["messages"][-1].content
            print(f"Assistant: {response_msg}\n")
        except GraphRecursionError:
            # Catch the limit gracefully if it hits a runaway loop
            print("\nAssistant: I'm sorry, I couldn't find any towns with ideal weather after checking several options.\n")


In [9]:
chat_loop()

UK Travel Assistant (type 'exit' to quit)


You:  Suggest two Cornwall beach towns with nice weather.






{'messages': [HumanMessage(content='Suggest two Cornwall beach towns with nice weather.', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-mlx', 'created_at': '2026-06-11T21:05:00.332951Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6028087458, 'load_duration': 1237689792, 'prompt_eval_count': 228, 'prompt_eval_duration': 518409000, 'eval_count': 130, 'eval_duration': 4270879000, 'logprobs': None, 'model_name': 'gemma4:12b-mlx', 'model_provider': 'ollama'}, id='lc_run--019eb880-921e-7632-bb8f-84534dbc47f7-0', tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'beach towns in Cornwall'}, 'id': '95621b2e-222e-4449-8dd3-fe0c80a1c90d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 228, 'output_tokens': 130, 'total_tokens': 358}), ToolMessage(content='Cornwall.jpg|300px]]\\&quot;}}&quot;}}">3</a></span> <span id="Falmouth" class="fn org listing-name"><a r

You:  exit
